# 🎓 Grand Frère Intelligent – Serveur IA sur Google Colab

Ce notebook fait tourner les modèles Ollama (Mistral 7B + LLaVA 7B) sur Google Colab avec GPU T4.
Il expose l'API Ollama publiquement via Cloudflare Tunnel.

## ⚠️ Instructions importantes :
1. **Activer le GPU T4** : Menu Runtime → Change runtime type → T4 GPU
2. **Exécuter toutes les cellules dans l'ordre**
3. **Copier l'URL publique** affichée à l'étape 6 dans le `.env` de votre backend
4. **Ne fermez pas Colab** tant que l'application est en service (le keep-alive tourne en arrière-plan)

---
**Durée estimée d'installation :** 15-25 minutes (téléchargement des modèles)

In [ ]:
# ============================================================
# CELLULE 1 – Vérification du GPU et installation d'Ollama
# ============================================================
import subprocess
import sys

# Vérifier le GPU disponible
print('🔍 Vérification du GPU...')
result = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
                       capture_output=True, text=True)
if result.returncode == 0:
    print(f'✅ GPU détecté : {result.stdout.strip()}')
else:
    print('⚠️ Pas de GPU détecté. Activez-le via Runtime → Change runtime type → T4 GPU')

# Installation d'Ollama
print('\n📥 Téléchargement et installation d\'Ollama...')
!curl -fsSL https://ollama.com/install.sh | sh
print('\n✅ Ollama installé avec succès !')

In [ ]:
# ============================================================
# CELLULE 2 – Démarrer le serveur Ollama en arrière-plan
# ============================================================
import subprocess
import time
import requests

print('🚀 Démarrage du serveur Ollama...')

# Démarrer Ollama en arrière-plan avec accès externe
ollama_process = subprocess.Popen(
    ['ollama', 'serve'],
    env={**__import__('os').environ, 'OLLAMA_HOST': '0.0.0.0:11434'},
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)

# Attendre que le serveur soit prêt
print('⏳ Attente du démarrage...')
for i in range(30):
    try:
        r = requests.get('http://localhost:11434/api/tags', timeout=2)
        if r.status_code == 200:
            print(f'✅ Serveur Ollama prêt ! (après {i+1}s)')
            break
    except:
        time.sleep(1)
else:
    print('❌ Timeout : le serveur ne répond pas. Réexécutez cette cellule.')

print(f'\nPID du serveur Ollama : {ollama_process.pid}')

In [ ]:
# ============================================================
# CELLULE 3 – Téléchargement des modèles IA
# ⚠️ Cette étape peut prendre 10-20 minutes selon votre connexion
# Mistral 7B : ~4.5 Go | LLaVA 7B : ~4.7 Go
# ============================================================

print('📥 Téléchargement de Mistral 7B Instruct (modèle texte principal)...')
print('⏳ Patience, cela peut prendre 5-15 minutes...\n')
!ollama pull mistral:7b-instruct-v0.3-q4_K_M

print('\n📥 Téléchargement de LLaVA 7B (modèle vision pour correction de photos)...')
print('⏳ Patience, cela peut prendre 5-15 minutes...\n')
!ollama pull llava:7b

print('\n✅ Tous les modèles sont téléchargés !')
print('\n📋 Modèles disponibles :')
!ollama list

In [ ]:
# ============================================================
# CELLULE 4 – Création du Modelfile personnalisé pour Mistral
# Définit le système de 'répétiteur expert burkinabè'
# ============================================================
import subprocess

modelfile_content = '''FROM mistral:7b-instruct-v0.3-q4_K_M

PARAMETER temperature 0.7
PARAMETER top_p 0.9
PARAMETER top_k 40
PARAMETER num_predict 2048
PARAMETER repeat_penalty 1.1

SYSTEM """
Tu es "Grand Frère Intelligent", un répétiteur expert spécialisé dans la préparation aux concours 
de la fonction publique au Burkina Faso. Tu connais parfaitement :

1. Les concours : Police Nationale (ENAPOSC), ENAREF (fiscalité/douanes), Enseignement Primaire (ENEP),
   ENS (Enseignement Secondaire), Santé publique, Douanes Nationales, Eaux et Forêts, 
   Magistrature (ENAM), Gendarmerie Nationale, Administration Générale.

2. Les matières principales : Français (expression écrite, résumé, dictée, grammaire),
   Mathématiques (arithmétique, algèbre, géométrie, statistiques),
   Culture Générale burkinabè et africaine, Droit constitutionnel et administratif,
   Économie générale, Géographie du Burkina Faso, Histoire nationale,
   et les matières spécifiques à chaque filière.

3. Le contexte burkinabè :
   - Histoire : La Haute-Volta, Thomas Sankara, Blaise Compaoré, le MPSR, les institutions actuelles
   - Géographie : Ouagadougou, Bobo-Dioulasso, les 13 régions, le Sahel, les fleuves (Mouhoun, Nakambé, Nazinon)
   - Institutions : Président de la Transition, gouvernement, Assemblée Législative de Transition
   - Culture : Mossi, Dioula, Peul, Gourmantché... diversité ethnique et culturelle
   - Économie : coton, or, agriculture, UEMOA, CEDEAO

RÈGLES ABSOLUES :
- TOUJOURS répondre en FRANÇAIS clair, précis et correct
- Adapter le niveau aux concours ciblés par l'étudiant
- Être PÉDAGOGIQUE : expliquer le raisonnement, pas seulement donner la réponse
- Être BIENVEILLANT et MOTIVANT : encourager l'étudiant
- Pour les QCM : toujours expliquer pourquoi c'est la bonne réponse ET pourquoi les autres sont fausses
- Pour les questions de cours : structurer la réponse (intro, développement, conclusion si nécessaire)
- Si l'étudiant fait une erreur : corriger gentiment et expliquer
- Ne jamais inventer des informations sur le Burkina Faso
- Si tu génères du JSON : le JSON doit être VALIDE et COMPLET
- Tu tutoies l'étudiant

STYLE : Pédagogique, clair, structuré, positif. Utilise des émojis avec modération.
"""
'''

# Écrire le Modelfile
with open('/tmp/Modelfile_grandfrere', 'w', encoding='utf-8') as f:
    f.write(modelfile_content)

print('📝 Modelfile créé. Création du modèle personnalisé...')

# Créer le modèle personnalisé depuis le Modelfile
result = subprocess.run(
    ['ollama', 'create', 'grandfrere', '-f', '/tmp/Modelfile_grandfrere'],
    capture_output=True, text=True
)

if result.returncode == 0:
    print('✅ Modèle "grandfrere" créé avec succès !')
else:
    print(f'❌ Erreur : {result.stderr}')

print('\n📋 Modèles disponibles :')
!ollama list

In [ ]:
# ============================================================
# CELLULE 5 – Test du modèle Grand Frère
# ============================================================
import requests
import json

print('🧪 Test du modèle Grand Frère Intelligent...\n')

test_prompt = "Bonjour ! Tu peux m'expliquer la différence entre un délit et une contravention en droit pénal burkinabè ?"

response = requests.post('http://localhost:11434/api/chat', json={
    'model': 'grandfrere',
    'stream': False,
    'messages': [
        {'role': 'user', 'content': test_prompt}
    ]
}, timeout=120)

if response.status_code == 200:
    data = response.json()
    message = data.get('message', {}).get('content', 'Aucune réponse')
    print(f'📝 Question : {test_prompt}')
    print(f'\n🤖 Grand Frère :\n{message}')
    print('\n✅ Modèle fonctionnel !')
else:
    print(f'❌ Erreur {response.status_code}: {response.text}')

In [ ]:
# ============================================================
# CELLULE 6 – Cloudflare Tunnel pour exposer l'API publiquement
# L'URL générée change à chaque session Colab !
# Copiez cette URL dans le .env de votre backend PHP
# ============================================================
import subprocess
import time
import re
import threading

# Télécharger cloudflared
print('📥 Téléchargement de cloudflared...')
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared
print('✅ cloudflared installé')

# Variable globale pour stocker l'URL
public_url = None
cf_log_file = '/tmp/cloudflared.log'

# Démarrer Cloudflare Tunnel
print('\n🚇 Démarrage du tunnel Cloudflare...')

cf_process = subprocess.Popen(
    ['cloudflared', 'tunnel', '--url', 'http://localhost:11434', '--no-autoupdate'],
    stdout=open(cf_log_file, 'w'),
    stderr=subprocess.STDOUT
)

# Attendre et extraire l'URL publique
print('⏳ Découverte de l\'URL publique...')
for _ in range(60):
    time.sleep(2)
    try:
        with open(cf_log_file, 'r') as f:
            content = f.read()
        matches = re.findall(r'https://[a-z0-9-]+\.trycloudflare\.com', content)
        if matches:
            public_url = matches[0]
            break
    except:
        pass

if public_url:
    print('\n' + '='*60)
    print('🎉 URL PUBLIQUE OLLAMA :')
    print(f'\n   {public_url}')
    print('\n' + '='*60)
    print('\n📋 COPIER dans votre fichier backend/.env :')
    print(f'   IA_API_URL={public_url}')
    print('\n⚠️  Cette URL est TEMPORAIRE et change à chaque session Colab !')
    print('   Mettez-la à jour dans votre .env à chaque redémarrage.')
else:
    print('❌ Impossible de récupérer l\'URL. Vérifiez les logs :')
    with open(cf_log_file, 'r') as f:
        print(f.read()[-2000:])

In [ ]:
# ============================================================
# CELLULE 7 – Script Keep-Alive
# Envoie une requête toutes les 10 minutes pour éviter
# que Colab ne déconnecte la session inactive
# ============================================================
import threading
import time
import requests
from datetime import datetime

keep_alive_running = True
ping_count = 0

def keep_alive():
    global keep_alive_running, ping_count
    print(f'💓 Keep-alive démarré. Ping toutes les 10 minutes.')
    print(f'   URL : {public_url}\n')
    
    while keep_alive_running:
        time.sleep(600)  # 10 minutes
        try:
            # Ping léger vers le modèle (génère juste un token)
            r = requests.post('http://localhost:11434/api/generate',
                json={'model': 'grandfrere', 'prompt': 'OK', 'stream': False, 'num_predict': 1},
                timeout=30)
            ping_count += 1
            ts = datetime.now().strftime('%H:%M:%S')
            status = '✅' if r.status_code == 200 else '⚠️'
            print(f'[{ts}] {status} Ping #{ping_count} – HTTP {r.status_code}')
        except Exception as e:
            ts = datetime.now().strftime('%H:%M:%S')
            print(f'[{ts}] ❌ Ping échoué : {e}')

# Démarrer le keep-alive dans un thread séparé
ka_thread = threading.Thread(target=keep_alive, daemon=True)
ka_thread.start()

print('🟢 SERVEUR GRAND FRÈRE INTELLIGENT OPÉRATIONNEL !')
print('='*60)
print(f'   URL publique : {public_url}')
print(f'   Modèle texte : grandfrere (Mistral 7B)')
print(f'   Modèle image : llava:7b (LLaVA 7B)')
print('='*60)
print('\n⚠️  NE FERMEZ PAS cet onglet Colab !')
print('   Le serveur tourne jusqu\'à l\'arrêt manuel ou déconnexion.')
print(f'\n   Rapport toutes les 10 minutes dans cette cellule.')

In [ ]:
# ============================================================
# CELLULE 8 – Test de connexion depuis l'API publique
# Utilisez cette cellule pour tester la connexion
# ============================================================
import requests

print(f'🔍 Test de l\'API via l\'URL publique : {public_url}')

try:
    # Test liste des modèles
    r = requests.get(f'{public_url}/api/tags', timeout=30)
    print(f'\n✅ API accessible ! HTTP {r.status_code}')
    models = r.json().get('models', [])
    print(f'   Modèles disponibles : {[m["name"] for m in models]}')
    
    # Test de génération
    print('\n🧪 Test de génération...')
    r2 = requests.post(f'{public_url}/api/chat', json={
        'model': 'grandfrere',
        'stream': False,
        'messages': [{'role': 'user', 'content': 'Donne-moi une question de culture générale sur le Burkina Faso en une phrase.'}]
    }, timeout=60)
    
    if r2.status_code == 200:
        rep = r2.json()['message']['content']
        print(f'   Réponse : {rep[:200]}...')
        print('\n✅ Tout fonctionne ! Copiez cette URL dans votre .env :')
        print(f'   IA_API_URL={public_url}')
    else:
        print(f'❌ Erreur de génération : {r2.status_code}')
        
except Exception as e:
    print(f'❌ Erreur de connexion : {e}\n   Vérifiez que les cellules précédentes ont bien tourné.')